# Physical evaluation viewer

Визуализация post-training evaluation для conditioned assimilation model. Ноутбук читает результаты `assim_lib.evaluate`, проверяет использованные `mean/std`, восстанавливает метрики семейства validation по каждой дате и показывает физические поля ансамбля.

Нужен запуск evaluation с `--save-ensembles`, потому что validation-compatible метрики по members вычисляются из сохраненных `.npz`.

In [ ]:
import csv
import json
import os
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

plt.style.use('seaborn-v0_8-whitegrid')

def _converted(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return value

def read_csv_rows(path):
    with open(path, newline='') as handle:
        return [{key: _converted(value) for key, value in row.items()} for row in csv.DictReader(handle)]

def rows_where(rows, **matches):
    return [row for row in rows if all(row.get(key) == value for key, value in matches.items())]

def values(rows, key):
    return np.asarray([row[key] for row in rows], dtype=np.float64)

def show_table(rows, columns=None, digits=5):
    rows = list(rows)
    if not rows:
        display(HTML('<em>No rows</em>'))
        return
    columns = columns or list(rows[0])
    def formatted(value):
        if isinstance(value, (float, np.floating)):
            return f'{value:.{digits}f}' if np.isfinite(value) else str(value)
        return str(value)
    head = ''.join(f'<th>{escape(str(column))}</th>' for column in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(formatted(row.get(column, "")))}</td>' for column in columns) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

## Path to evaluation output

Укажите директорию результата evaluation. На сервере она обычно имеет вид:

```text
/home/checkpoints/concat_conditioning/m2m_2f/<run_name>/evaluation/valid_physical_15d_15ens
```

Путь также можно передать переменной окружения `ASSIM_EVAL_DIR` перед запуском Jupyter.

In [ ]:
EVAL_DIR = Path(os.environ.get(
    'ASSIM_EVAL_DIR',
    '/home/checkpoints/concat_conditioning/m2m_2f/<run_name>/evaluation/valid_physical_15d_15ens',
))

required = ['metadata.json', 'aggregate_metrics.csv', 'per_case_metrics.csv']
missing = [name for name in required if not (EVAL_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing} in EVAL_DIR={EVAL_DIR}. Set EVAL_DIR to your evaluation output.')

metadata = json.loads((EVAL_DIR / 'metadata.json').read_text())
aggregate = read_csv_rows(EVAL_DIR / 'aggregate_metrics.csv')
per_case = read_csv_rows(EVAL_DIR / 'per_case_metrics.csv')
sample_paths = sorted((EVAL_DIR / 'samples').glob('*.npz'))

print('EVAL_DIR:', EVAL_DIR)
print('Saved case arrays:', len(sample_paths))
if not sample_paths:
    raise FileNotFoundError('No samples/*.npz found. Re-run evaluation with --save-ensembles.')

## Evaluation setup and normalization

`normalization_means` и `normalization_stds` ниже - именно те значения, которыми evaluation денормализовал output, truth и background в физические единицы.

In [ ]:
setup = {
    'checkpoint': metadata.get('checkpoint'),
    'data_config': metadata.get('data_config'),
    'split': metadata.get('split'),
    'num_cases': metadata.get('num_cases'),
    'stride_days': metadata.get('stride_days'),
    'ensemble_size': metadata.get('ensemble_size'),
    'sample_batch_size': metadata.get('sample_batch_size'),
    'inference_precision': metadata.get('inference_precision'),
    'means_used': metadata.get('normalization_means'),
    'stds_used': metadata.get('normalization_stds'),
    'concentration_clipping': metadata.get('concentration_clipping'),
}
show_table([{'parameter': key, 'value': value} for key, value in setup.items()], ['parameter', 'value'])

case_info = metadata.get('cases', [])
columns = [c for c in ['case_order', 'target_date', 'background_date', 'background_offset_days', 'hour', 'conditioning_kind', 'obs_count', 'observed_fraction'] if any(c in row for row in case_info)]
show_table(case_info, columns or None)

## Metrics recorded during training validation

Этот блок читает `metrics.json` из директории training run и показывает **ровно те значения, которые были сохранены на каждой validation epoch**. Они считаются training pipeline в нормализованном пространстве и с настройками ensemble/stride, использованными во время обучения.

In [ ]:
RUN_DIR = EVAL_DIR.parent.parent
training_metrics_path = RUN_DIR / 'metrics.json'
if training_metrics_path.exists():
    training_history = json.loads(training_metrics_path.read_text())
    print('Training metrics:', training_metrics_path)
    history_columns = [c for c in [
        'epoch', 'val_loss', 'val_loss_full', 'val_loss_obs',
        'background_rmse_full', 'analysis_rmse_mean_full', 'analysis_rmse_of_mean_full',
        'analysis_rmse_skill_mean_full', 'analysis_rmse_skill_of_mean_full',
        'background_rmse_obs', 'analysis_rmse_mean_obs', 'analysis_rmse_of_mean_obs',
        'analysis_rmse_skill_mean_obs', 'analysis_rmse_skill_of_mean_obs',
    ] if any(c in row for row in training_history)]
    show_table(training_history, history_columns, digits=6)
    plot_columns = [c for c in ['background_rmse_full', 'analysis_rmse_mean_full', 'analysis_rmse_of_mean_full'] if any(c in row for row in training_history)]
    skill_columns = [c for c in ['analysis_rmse_skill_mean_full', 'analysis_rmse_skill_of_mean_full'] if any(c in row for row in training_history)]
    if plot_columns or skill_columns:
        fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
        x = np.asarray([row.get('epoch', index) for index, row in enumerate(training_history)])
        for name in plot_columns:
            axes[0].plot(x, [row.get(name, np.nan) for row in training_history], marker='o', label=name)
        axes[0].set_title('Saved validation RMSE (full)')
        axes[0].set_xlabel('epoch')
        axes[0].legend()
        axes[1].axhline(0, color='black', linewidth=1)
        for name in skill_columns:
            axes[1].plot(x, [row.get(name, np.nan) for row in training_history], marker='o', label=name)
        axes[1].set_title('Saved validation RMSE skill (full)')
        axes[1].set_xlabel('epoch')
        axes[1].legend()
        plt.show()
else:
    print(f'No training metrics found at {training_metrics_path}; continuing with post-training physical evaluation.')

## Validation-compatible metrics in physical units

Во время обучения `compute_sample_validation_metrics` вычисляет для `full` и `obs`:

- `background_mae`, `background_rmse`;
- `analysis_mae_mean/min/max/of_mean`;
- `analysis_rmse_mean/min/max/of_mean`;
- `analysis_rmse_skill_mean/min/max/of_mean`.

Ниже те же определения считаются на сохраненных результатах evaluation, но **в физических единицах после clipping концентрации**. Регион `obs` соответствует `obs_mask`.

Примечание: текущий training-код сохранял `analysis_rmse_skill_min/max` с перепутанными именами; здесь `min` и `max` подписаны корректно.

In [ ]:
def _case_order(path):
    return int(path.name.split('_', 1)[0])

case_by_order = {int(c['case_order']): c for c in metadata.get('cases', [])}

def _error_metrics(pred, truth, mask):
    selector = np.asarray(mask, dtype=bool)
    if not selector.any():
        return np.nan, np.nan, 0
    diff = np.asarray(pred, dtype=np.float64)[selector] - np.asarray(truth, dtype=np.float64)[selector]
    return float(np.abs(diff).mean()), float(np.sqrt(np.square(diff).mean())), int(selector.sum())

def _skill(rmse, background_rmse):
    return np.nan if not np.isfinite(background_rmse) or background_rmse <= 0 else 1.0 - rmse / background_rmse

def _row_from_case(path, region):
    with np.load(path) as payload:
        samples = payload['analysis_ensemble'].astype(np.float64)
        truth = payload['truth'].astype(np.float64)
        background = payload['background'].astype(np.float64)
        mask = (payload['valid_mask'] if region == 'full' else payload['obs_mask']).astype(bool)
    background_mae, background_rmse, count = _error_metrics(background, truth, mask)
    if count == 0:
        return None
    member_mae, member_rmse = [], []
    for member in samples:
        mae, rmse, _ = _error_metrics(member, truth, mask)
        member_mae.append(mae)
        member_rmse.append(rmse)
    mean_mae, mean_rmse, _ = _error_metrics(samples.mean(axis=0), truth, mask)
    member_mae = np.asarray(member_mae)
    member_rmse = np.asarray(member_rmse)
    member_skill = 1.0 - member_rmse / background_rmse if background_rmse > 0 else np.full_like(member_rmse, np.nan)
    order = _case_order(path)
    info = case_by_order.get(order, {})
    return {
        'case_order': order,
        'target_date': info.get('target_date', info.get('case_id', path.stem)),
        'region': region,
        'metric_count': count,
        'background_mae': background_mae,
        'background_rmse': background_rmse,
        'analysis_mae_mean': float(member_mae.mean()),
        'analysis_mae_min': float(member_mae.min()),
        'analysis_mae_max': float(member_mae.max()),
        'analysis_mae_of_mean': mean_mae,
        'analysis_rmse_mean': float(member_rmse.mean()),
        'analysis_rmse_min': float(member_rmse.min()),
        'analysis_rmse_max': float(member_rmse.max()),
        'analysis_rmse_of_mean': mean_rmse,
        'analysis_rmse_skill_mean': float(np.nanmean(member_skill)),
        'analysis_rmse_skill_min': float(np.nanmin(member_skill)),
        'analysis_rmse_skill_max': float(np.nanmax(member_skill)),
        'analysis_rmse_skill_of_mean': _skill(mean_rmse, background_rmse),
    }

validation_rows = []
for path in sample_paths:
    for region in ('full', 'obs'):
        row = _row_from_case(path, region)
        if row is not None:
            validation_rows.append(row)
validation_by_date = sorted(validation_rows, key=lambda row: (row['region'], row['case_order']))
for row in validation_by_date:
    row['date_label'] = str(row['target_date'])
date_columns = [
    'target_date', 'region', 'background_rmse', 'analysis_rmse_mean',
    'analysis_rmse_of_mean', 'analysis_rmse_skill_mean', 'analysis_rmse_skill_of_mean',
]
show_table(validation_by_date, date_columns)

In [ ]:
def _empty_sums():
    return {'abs': 0.0, 'sq': 0.0, 'count': 0}

def _add_sums(totals, pred, truth, mask):
    selector = np.asarray(mask, dtype=bool)
    diff = np.asarray(pred, dtype=np.float64)[selector] - np.asarray(truth, dtype=np.float64)[selector]
    totals['abs'] += float(np.abs(diff).sum())
    totals['sq'] += float(np.square(diff).sum())
    totals['count'] += int(selector.sum())

def _finish(totals):
    if totals['count'] == 0:
        return np.nan, np.nan
    return totals['abs'] / totals['count'], np.sqrt(totals['sq'] / totals['count'])

def pooled_validation_summary(paths, region):
    ensemble_size = int(metadata['ensemble_size'])
    bg_total = _empty_sums()
    member_totals = [_empty_sums() for _ in range(ensemble_size)]
    mean_total = _empty_sums()
    for path in paths:
        with np.load(path) as payload:
            samples = payload['analysis_ensemble']
            truth = payload['truth']
            background = payload['background']
            mask = (payload['valid_mask'] if region == 'full' else payload['obs_mask']).astype(bool)
        if not mask.any():
            continue
        _add_sums(bg_total, background, truth, mask)
        _add_sums(mean_total, samples.mean(axis=0), truth, mask)
        for index, member in enumerate(samples):
            _add_sums(member_totals[index], member, truth, mask)
    background_mae, background_rmse = _finish(bg_total)
    member_metrics = np.asarray([_finish(total) for total in member_totals])
    mean_mae, mean_rmse = _finish(mean_total)
    member_skill = 1.0 - member_metrics[:, 1] / background_rmse if background_rmse > 0 else np.full(ensemble_size, np.nan)
    return {
        'region': region,
        'metric_count': bg_total['count'],
        'background_mae': background_mae,
        'background_rmse': background_rmse,
        'analysis_mae_mean': member_metrics[:, 0].mean(),
        'analysis_mae_min': member_metrics[:, 0].min(),
        'analysis_mae_max': member_metrics[:, 0].max(),
        'analysis_mae_of_mean': mean_mae,
        'analysis_rmse_mean': member_metrics[:, 1].mean(),
        'analysis_rmse_min': member_metrics[:, 1].min(),
        'analysis_rmse_max': member_metrics[:, 1].max(),
        'analysis_rmse_of_mean': mean_rmse,
        'analysis_rmse_skill_mean': np.nanmean(member_skill),
        'analysis_rmse_skill_min': np.nanmin(member_skill),
        'analysis_rmse_skill_max': np.nanmax(member_skill),
        'analysis_rmse_skill_of_mean': _skill(mean_rmse, background_rmse),
    }

validation_summary = [pooled_validation_summary(sample_paths, region) for region in ('full', 'obs')]
show_table(validation_summary)

## Validation metrics by date

На графиках отображаются метрики, сопоставимые с sample-validation: RMSE background, RMSE отдельного member в среднем и RMSE ensemble mean; ниже - RMSE skill относительно background.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 9), constrained_layout=True)
for col, region in enumerate(('full', 'obs')):
    view = rows_where(validation_by_date, region=region)
    x = np.arange(len(view))
    axes[0, col].plot(x, values(view, 'background_rmse'), marker='o', label='background RMSE')
    axes[0, col].plot(x, values(view, 'analysis_rmse_mean'), marker='o', label='member RMSE mean')
    axes[0, col].plot(x, values(view, 'analysis_rmse_of_mean'), marker='o', label='ensemble mean RMSE')
    axes[0, col].set_title(f'RMSE, region={region}')
    axes[0, col].set_ylabel('physical RMSE')
    axes[0, col].legend()
    axes[1, col].axhline(0, color='black', linewidth=1)
    axes[1, col].plot(x, values(view, 'analysis_rmse_skill_mean'), marker='o', label='member skill mean')
    axes[1, col].plot(x, values(view, 'analysis_rmse_skill_of_mean'), marker='o', label='ensemble mean skill')
    axes[1, col].set_title(f'RMSE skill vs background, region={region}')
    axes[1, col].set_ylabel('1 - RMSE(analysis) / RMSE(background)')
    axes[1, col].set_xticks(x)
    axes[1, col].set_xticklabels([row['date_label'] for row in view], rotation=60, ha='right')
    axes[1, col].legend()
plt.show()

## Physical per-field metrics

Standalone evaluation дополнительно сохраняет метрики отдельно по физическим переменным (`siconc`, `sithic`, ...). Это удобнее для интерпретации, чем training validation, которая агрегировала каналы в каждом регионе.

In [ ]:
aggregate_columns = ['field', 'region', 'analysis_mean_rmse', 'background_rmse', 'analysis_rmse_skill', 'analysis_crps', 'analysis_spread', 'analysis_spread_skill_ratio']
aggregate_view = [row for row in aggregate if row['region'] in ('full', 'observed')]
show_table(aggregate_view, aggregate_columns)

fields = list(dict.fromkeys(row['field'] for row in per_case))
fig, axes = plt.subplots(len(fields), 2, figsize=(18, 4 * len(fields)), squeeze=False, constrained_layout=True)
for row_index, field in enumerate(fields):
    view = sorted(rows_where(per_case, field=field, region='full'), key=lambda row: row['case_order'])
    x = np.arange(len(view))
    axes[row_index, 0].plot(x, values(view, 'background_rmse'), label='background', marker='o')
    axes[row_index, 0].plot(x, values(view, 'analysis_mean_rmse'), label='ensemble mean', marker='o')
    axes[row_index, 0].set_title(f'{field}: RMSE (full)')
    axes[row_index, 0].legend()
    axes[row_index, 1].axhline(0, color='black', linewidth=1)
    axes[row_index, 1].plot(x, values(view, 'analysis_rmse_skill'), marker='o')
    axes[row_index, 1].set_title(f'{field}: ensemble mean RMSE skill (full)')
    for ax in axes[row_index]:
        ax.set_xticks(x)
        labels = [str(item.get('target_date', item.get('case_id'))) for item in view]
        ax.set_xticklabels(labels, rotation=60, ha='right')
plt.show()

## Selected case maps

Укажите `CASE_ORDER` и `FIELD`. Карты уже находятся в физических координатах; для `siconc` шкала фиксирована в `[0, 1]`.

In [ ]:
CASE_ORDER = 0
FIELD = fields[0]  # e.g. 'siconc' or 'sithic'

selected_path = next(path for path in sample_paths if _case_order(path) == CASE_ORDER)
with np.load(selected_path) as payload:
    case_fields = payload['fields'].astype(str).tolist()
    channel = case_fields.index(FIELD)
    samples = payload['analysis_ensemble'].astype(np.float64)
    truth = payload['truth'].astype(np.float64)
    background = payload['background'].astype(np.float64)
    obs_values = payload['obs_values'].astype(np.float64)
    obs_mask = payload['obs_mask'].astype(bool)
    valid_mask = payload['valid_mask'].astype(bool)

analysis_mean = samples.mean(axis=0)
analysis_std = samples.std(axis=0, ddof=1 if samples.shape[0] > 1 else 0)
mask = valid_mask[channel]
case_label = case_by_order.get(CASE_ORDER, {}).get('target_date', selected_path.stem)

def masked(values, selector=mask):
    return np.where(selector, values, np.nan)

if FIELD == 'siconc':
    limits = (0.0, 1.0)
else:
    joined = np.concatenate([background[channel][mask], truth[channel][mask], analysis_mean[channel][mask]])
    limits = tuple(np.nanpercentile(joined, [1, 99]))
err = analysis_mean[channel] - truth[channel]
err_lim = np.nanpercentile(np.abs(err[mask]), 99) if mask.any() else 1.0

panels = [
    ('Background', masked(background[channel]), 'viridis', limits),
    ('Truth', masked(truth[channel]), 'viridis', limits),
    ('Ensemble mean', masked(analysis_mean[channel]), 'viridis', limits),
    ('Mean - truth', masked(err), 'RdBu_r', (-err_lim, err_lim)),
    ('Ensemble std', masked(analysis_std[channel]), 'magma', (0, np.nanpercentile(analysis_std[channel][mask], 99))),
    ('Condition', np.where(obs_mask[channel] & mask, obs_values[channel], np.nan), 'viridis', limits),
]
fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
for ax, (title, values, cmap, (vmin, vmax)) in zip(axes.ravel(), panels):
    im = ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f'{case_label} | {FIELD} | {samples.shape[0]} ensemble members', fontsize=15)
plt.show()

show_table(rows_where(per_case, case_order=float(CASE_ORDER), field=FIELD))

## Individual ensemble members for selected case

Эта панель позволяет увидеть разнообразие 15 samples; основная reported analysis остается ensemble mean.

In [ ]:
N_SHOW = min(15, samples.shape[0])
ncols = 5
nrows = int(np.ceil(N_SHOW / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(17, 3.5 * nrows), squeeze=False, constrained_layout=True)
for index, ax in enumerate(axes.ravel()):
    if index >= N_SHOW:
        ax.axis('off')
        continue
    values = masked(samples[index, channel])
    im = ax.imshow(values, cmap='viridis', vmin=limits[0], vmax=limits[1])
    ax.set_title(f'Member {index:02d}')
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
fig.suptitle(f'{case_label} | {FIELD} | individual physical samples', fontsize=15)
plt.show()